# behavior-diff quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertlieyingadrian/behavior-diff/blob/main/examples/quickstart.ipynb)

**What did my fine-tune change?** `behavior-diff` samples a base model and a fine-tune on the same prompts, has a
judge model describe how each pair of replies differs, groups the descriptions into named behaviors, and tests each
one with a sign test against a same-model control (the base against itself).

This notebook replays a real run from saved data. It needs no API key, no GPU and no network after the install:

- the fine-tune: a Qwen3-8B LoRA trained on Tinker for 30 steps to answer in bullet points;
- 128 prompts from the default suite, the base sampled twice and the fine-tune once;
- every judge reply, cached.

In [ ]:
!pip install -q behavior-diff || pip install -q "git+https://github.com/albertlieyingadrian/behavior-diff"

## 1. Run the demo

`bdiff demo` in a terminal does the same thing.

In [ ]:
from bdiff import demo

demo.main(["-o", "demo"])

## 2. Read the result

Each category has two counts: on how many prompts the judge saw it more in the fine-tune (`n_new`) and more in the
base (`n_ref`). A category has **changed** when a sign test on those two counts passes after Benjamini-Hochberg
correction (q < 0.05). The same-model control runs the same test on the base against itself; it should show 0.

In [ ]:
import json

rep = json.load(open("demo/report.json"))
print(f"{'category':38} {'tuned':>5} {'base':>5} {'q':>7}  changed")
for r in rep["main"]["categories"]:
    print(f"{r['name'][:38]:38} {r['n_new']:>5} {r['n_ref']:>5} {r['q']:>7.3f}  {'yes' if r['changed'] else ''}")

ctrl = rep["control"]["categories"]
print(f"\nsame-model control: {sum(r['changed'] for r in ctrl)} of {len(ctrl)} categories changed")

The bullet-point LoRA comes out as **"Uses concise response style"** (25 prompts against 10, q = 0.047), with lost
detail, examples and empathy. Measured directly, 125 of 128 fine-tune replies were bullet lists against 2 of 128 for
the base. The base already uses markdown lists, so the judge named the change as a terser style rather than as a format.

## 3. Pre-registered vs post hoc

The prediction was written before the run: *a changed category about bullet or list formatting, toward the
fine-tune*. The strict score matches the category name; a match only in its definition is a weak hit.

In [ ]:
for s in rep["expectations"]["scores"]:
    pre, post = s["preregistered"], s["posthoc"]
    print(s["id"], "-", s["claim"])
    print("  pre-registered:", "hit" if pre["hit"] else "miss", f"(matched in the {pre['strength']})" if pre["hit"] else "")
    print("  post hoc:      ", "hit" if post["hit"] else "miss", f"({post['reading']})" if post["hit"] else "")

## 4. The HTML report

In [ ]:
try:
    from IPython.display import HTML, display

    display(HTML(open("demo/report.html").read()))
except ImportError:
    print("open demo/report.html in a browser")

## 5. Your own models

These need keys and cost money, so they are not run here. The judge defaults to an open-weight model on OpenRouter;
`--judge-url` takes any OpenAI-compatible endpoint, including your own vLLM server.

```bash
# a Tinker checkpoint against its own base model
export TINKER_API_KEY=... OPENROUTER_API_KEY=...
bdiff run --tuned tinker://<run-id>/sampler_weights/final

# a LoRA adapter from the Hugging Face Hub, sampled with vLLM on a Modal GPU
pip install "behavior-diff[modal]" && modal setup
bdiff run --base modal:Qwen/Qwen2.5-7B-Instruct --tuned modal:Qwen/Qwen2.5-7B-Instruct+<owner>/<adapter>

# drift across generations of a self-improvement loop
bdiff generations tinker://.../gen0 tinker://.../gen1 tinker://.../gen2 tinker://.../gen3
```

Docs: https://behavior-diff.mintlify.site